In [ ]:
# CELL 1: Imports e configuração
import logging
import pandas as pd

from functions.config import ProjectConfig
from functions.Diretories_Downloads import cif_download
from functions.Primary_Filter import primary_filter
from functions.Extraction_Phases import extrair_fracoes_fase
from functions.Plots import plot_filtro_primario, plot_rietveld, plot_refinamento_com_referencias

# Configura logging para exibir mensagens de INFO e acima no notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
# CELL 2: Cria o diretório do projeto

cfg = ProjectConfig("projeto_x")
cfg.create_dirs()

print(cfg.project_dir)
print(cfg.ref_dir)
print(cfg.results_dir)

In [ ]:
# CELL 3: Download referências

dict_refs = {
    # alpha-Fe2O3: Trigonal, R-3c. Referência clássica de alta cristalinidade.
    "Hematite": 2101168,

    # Fe3O4: Cúbico, Fd-3m. Estrutura de espinélio invertido.
    # Cuidado: Picos muito próximos aos da Maghemita.
    "Magnetite": 9005813,

    # gamma-Fe2O3: Cúbico, P4332. Fase metaestável.
    # Importante conferir se não há picos de superestrutura no seu DRX.
    "Maghemite": 9017489,

    # alpha-FeOOH: Ortorrômbico, Pnma. Comum se houver hidratação no resíduo.
    "Goethite": 7720752,

    # FeO: Cúbico, Fm-3m. FASE CRÍTICA para a redução a 850 °C.
    "Wüstite": 9002673,

    # Fe3C: Ortorrômbico, Pnma. Fase crítica para a redução a 850 °C.
    "Cementite": 9016584,

    # alpha-Fe: Cúbico, Im-3m. O produto final da redução (Ferro metálico).
    "Fe_metalico": 9008536,
}

for Nome, cod_ID in dict_refs.items():
    cif_download(Nome, cod_ID, str(cfg.ref_dir))

In [ ]:
# CELL 4: Filtragem primária

input_drx_raw = "inputs/hematite_raw.txt"
df, amostra_norm, theta = primary_filter(
    correlation="pearson",
    input_file=input_drx_raw,
    ref_dir=str(cfg.ref_dir),
)
print(df)

In [ ]:
# CELL 5: Plot filtro primário (amostra vs melhor referência)

plot_filtro_primario(theta, amostra_norm, df)

In [ ]:
# CELL 6: Refinamento Rietveld (GSAS-II)

import importlib
import functions.Workflow_GSAS2 as wf
importlib.reload(wf)
from functions.Workflow_GSAS2 import refinamento_sequencial_oxidos

refs_cif = [
    str(cfg.ref_dir / df['nome'][0]),
    str(cfg.ref_dir / df['nome'][1]),
    # str(cfg.ref_dir / df['nome'][2]),
    # str(cfg.ref_dir / df['nome'][3]),
]

input_inst = "inputs/inst_rruff.prm"
resultado = refinamento_sequencial_oxidos(
    input_drx_raw,
    input_inst,
    refs_cif,
    cfg.project_name,
    refinar_atomos=True,  # mude para False se o XRD for de baixa qualidade
)

In [ ]:
# CELL 7: Plot do Refinamento de Rietveld

plot_rietveld(resultado)

In [ ]:
# CELL 7b: Plot do refinamento com as curvas referenciais sobrepostas

plot_refinamento_com_referencias(
    resultado["x"],
    resultado["yobs"],
    resultado["ycalc"],
    refs_cif,
    theta,
)

In [ ]:
# CELL 8: Extração automática dos percentuais de fase do .lst

lst_file_path = str(cfg.results_dir / f"{cfg.project_name}.lst")

dados_extraidos = extrair_fracoes_fase(lst_file_path)

print("--- Resultados do Refinamento ---")
if isinstance(dados_extraidos, dict):
    for fase, fracoes in dados_extraidos.items():
        print(f"\nFase: {fase}")
        print(f"  Fração da Fase (Phase Fraction): {fracoes['Phase Fraction']}")

        weight_pct = fracoes['Weight Fraction'] * 100
        print(f"  Fração em Peso (Weight Fraction): {fracoes['Weight Fraction']} ({weight_pct:.2f}%)")
else:
    print(dados_extraidos)